In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from patsy import bs
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index

warnings.filterwarnings('ignore')

# =====================================================================
# 1. SETUP
# =====================================================================
sns.set_theme(style="whitegrid", rc={"grid.color": "#f0f0f0", "grid.linestyle": "-"})
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['pdf.fonttype'] = 42

# Okabe-Ito Colorblind-Safe Palette
surv_colors = {
    'Low Risk Profile': '#0072B2',   # Okabe-Ito Blue
    'High Risk Profile': '#D55E00'   # Okabe-Ito Vermilion
}
c_neutral = "#222222"


MODEL_FORMULA = "pHSC_z + coef_z"

# =====================================================================
# 3. DATA LOADING & PREPARATION 
# =====================================================================
moffitt_params = pd.read_csv("IntermediateResults/moffitt_weighted_params.csv")
moffitt_labs = pd.read_excel("InputData/moffitt_cohort_OS_labs.xlsx")
merged_moffitt = pd.merge(moffitt_labs, moffitt_params, left_on="Subject_ID", right_on="ID")

venex_labs = pd.read_excel("InputData/venex_cohort_OS_labs_with_mutations.xlsx")
venex_params = pd.read_csv("IntermediateResults/venex_weighted_params.csv")
merged_venex = pd.merge(venex_labs, venex_params, left_on="Study_subject", right_on="ID")

column_mapping_moffitt = {
    'OS_time': 'OS (days)',
    'OS_status': 'Vital_status',
    'biomarker_1': 'pHSC_weighted',
    'biomarker_2': 'coef_weighted'
}

df_moffitt = pd.DataFrame({
    'Patient_ID': merged_moffitt['Subject_ID'].astype(str).str.strip(),
    'OS_time': pd.to_numeric(merged_moffitt[column_mapping_moffitt['OS_time']], errors='coerce'),
    'OS_status': pd.to_numeric(merged_moffitt[column_mapping_moffitt['OS_status']], errors='coerce').fillna(0).astype(int),
    'pHSC_raw': pd.to_numeric(merged_moffitt[column_mapping_moffitt['biomarker_1']], errors='coerce'),
    'coef_raw': pd.to_numeric(merged_moffitt[column_mapping_moffitt['biomarker_2']], errors='coerce')
}).dropna().reset_index(drop=True)

df_venex = pd.DataFrame({
    'Patient_ID': merged_venex['Study_subject'].astype(str).str.strip(),
    'OS_time': pd.to_numeric(merged_venex['OS_time'], errors='coerce'),
    'OS_status': pd.to_numeric(merged_venex['OS_status'], errors='coerce').fillna(0).astype(int),
    'pHSC_raw': pd.to_numeric(merged_venex['pHSC_weighted'], errors='coerce'),
    'coef_raw': pd.to_numeric(merged_venex['coef_weighted'], errors='coerce')
}).dropna().reset_index(drop=True)

for df in [df_moffitt, df_venex]:
    for param in ['pHSC', 'coef']:
        raw_col = f"{param}_raw"
        z_col = f"{param}_z"
        m = df[raw_col].mean()
        s = df[raw_col].std() if df[raw_col].std() > 0 else 1.0
        df[z_col] = (df[raw_col] - m) / s

print(f"Loaded Moffitt Cohort: {len(df_moffitt)} patients ({int(df_moffitt['OS_status'].sum())} events)")
print(f"Loaded VENEX Cohort:   {len(df_venex)} patients ({int(df_venex['OS_status'].sum())} events)")

# =====================================================================
# 4. OPTIMAL CUTPOINT & STRATIFICATION 
# =====================================================================
def stratify_by_risk(df, score_col, t_col, e_col):
    scores = sorted(df[score_col].unique())
    if len(scores) < 3:
        cut = np.median(scores) if scores else 0.0
    else:
        low_b, high_b = np.percentile(scores, 15), np.percentile(scores, 85)
        candidates = [s for s in scores if low_b <= s <= high_b] or scores
        best_stat, cut = -1, np.median(scores)
        for c in candidates:
            grp = np.where(df[score_col] <= c, 0, 1)
            if len(np.unique(grp)) < 2:
                continue
            try:
                res = logrank_test(
                    df.loc[grp == 0, t_col], df.loc[grp == 1, t_col],
                    df.loc[grp == 0, e_col], df.loc[grp == 1, e_col]
                )
                if res.test_statistic > best_stat:
                    best_stat, cut = res.test_statistic, c
            except Exception:
                continue

    group_below = df[score_col] <= cut
    km_below = KaplanMeierFitter().fit(df.loc[group_below, t_col], df.loc[group_below, e_col])
    km_above = KaplanMeierFitter().fit(df.loc[~group_below, t_col], df.loc[~group_below, e_col])
    
    # Assign Low Risk to whichever subset exhibits superior median survival
    if km_below.median_survival_time_ >= km_above.median_survival_time_:
        df['final_group'] = np.where(group_below, 'Low Risk Profile', 'High Risk Profile')
    else:
        df['final_group'] = np.where(group_below, 'High Risk Profile', 'Low Risk Profile')

    res = logrank_test(
        df.loc[df['final_group'] == 'Low Risk Profile', t_col],
        df.loc[df['final_group'] == 'High Risk Profile', t_col],
        df.loc[df['final_group'] == 'Low Risk Profile', e_col],
        df.loc[df['final_group'] == 'High Risk Profile', e_col]
    )
    return cut, res.p_value

# =====================================================================
# 5. EXPERIMENTS SETUP
# =====================================================================
experiments = [
    {
        "panel_letter": "A",
        "title": "Moffitt Internal Baseline",
        "subtitle": "Train Moffitt → Test Moffitt (Direct Fit)",
        "train_df": df_moffitt.copy(),
        "test_df": df_moffitt.copy(),
    },
    {
        "panel_letter": "B",
        "title": "External Generalization to Moffitt",
        "subtitle": "Train Entire VENEX → Test Entire Moffitt",
        "train_df": df_venex.copy(),
        "test_df": df_moffitt.copy(),
    },
    {
        "panel_letter": "C",
        "title": "Reverse Generalization to VENEX",
        "subtitle": "Train Entire Moffitt → Test Entire VENEX",
        "train_df": df_moffitt.copy(),
        "test_df": df_venex.copy(),
    }
]

# =====================================================================
# 6. FIGURE RENDERING 
# =====================================================================
fig, axes = plt.subplots(1, 3, figsize=(24, 7.5), sharey=True)

for ax, exp in zip(axes, experiments):
    train_data = exp["train_df"]
    test_data = exp["test_df"]
    
    for param_col in ['pHSC_z', 'coef_z']:
        test_data[param_col] = np.clip(
            test_data[param_col], 
            train_data[param_col].min(), 
            train_data[param_col].max()
        )
    
    cph = CoxPHFitter(penalizer=0.01)
    cph.fit(train_data, duration_col='OS_time', event_col='OS_status', formula=MODEL_FORMULA)
    
    test_data['score_mech'] = cph.predict_log_partial_hazard(test_data)
    
    c_idx = concordance_index(test_data['OS_time'], -test_data['score_mech'], test_data['OS_status'])
    cut_point, p_val = stratify_by_risk(test_data, 'score_mech', 'OS_time', 'OS_status')

    for name in ['Low Risk Profile', 'High Risk Profile']:
        mask = test_data['final_group'] == name
        if mask.sum() == 0: 
            continue
        kmf = KaplanMeierFitter()
        kmf.fit(
            durations=test_data.loc[mask, 'OS_time'], 
            event_observed=test_data.loc[mask, 'OS_status'], 
            label=f"{name} (N = {mask.sum()})"
        )
        kmf.plot_survival_function(
            ax=ax,
            color=surv_colors[name],
            linestyle="-" if name == 'Low Risk Profile' else "--",
            linewidth=3.2,
            ci_show=False,
            show_censors=True,
            censor_styles={'ms': 7, 'marker': '+' if name == 'Low Risk Profile' else 'x'}
        )
        
    ax.set_xlabel('Time Since Treatment Initiation (Days)', fontsize=15, fontweight='bold', labelpad=8)
    ax.set_xlim(0, test_data['OS_time'].max() * 1.03)
    ax.set_ylim(-0.02, 1.02)
    ax.tick_params(axis='both', which='major', labelsize=13)
    
    # Title & Subplot 
    ax.text(-0.02, 1.05, exp["panel_letter"], transform=ax.transAxes, fontsize=24, fontweight='bold', va='top', ha='right')
    ax.set_title(exp["title"], fontsize=15, fontweight='bold', pad=14, loc='left', color=c_neutral)
    
    # Statistical Summary 
    p_string = f"Log-rank $p = {p_val:.4e}$" if p_val < 0.001 else f"Log-rank $p = {p_val:.4f}$"
    badge_text = (
        f"{exp['subtitle']}\n\n"
        f"C-Index = {c_idx:.3f}\n"
        f"{p_string}"
    )
    ax.text(0.48, 0.60, badge_text, transform=ax.transAxes, fontsize=12.5, fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.92, edgecolor='#cccccc', boxstyle='round,pad=0.5'))
    
    ax.legend(loc='lower left', frameon=True, fontsize=12.5, facecolor='white', edgecolor='#cccccc', borderpad=0.5)

axes[0].set_ylabel('Overall Survival Probability', fontsize=16, fontweight='bold', labelpad=10)

plt.tight_layout()
os.makedirs('Figures', exist_ok=True)
plt.savefig('Figures/Figure_Moffitt_DirectFit_BothParams_1x3.pdf', dpi=300, bbox_inches='tight')
plt.savefig('Figures/Figure_Moffitt_DirectFit_BothParams_1x3.png', dpi=300, bbox_inches='tight')
print("Saved direct-fit validation figure to 'Figures/Figure_Moffitt_DirectFit_BothParams_1x3.pdf'.")
plt.show()